# Korea RIM Valuation **v6** — 단일 계산 경로 + 밸류 폭발 구조적 차단

**v5 문제 (A326030)**

| 증상 | 원인 |
|---|---|
| 같은 종목 TP 가 셀마다 다름 (Cell 5: +344.6%, Cell 7: +2,513.7%) | Cell 5 테스트는 v5 패치(Cell 5-B) **이전** 클래스로, Cell 7 은 **이후** 클래스로 계산 → 몽키패치 순서에 따라 모형이 바뀜 |
| NPM 75.5% | `NI = a + b·Sales` 회귀의 **기울기 b(한계이익률)** 를 평균 NPM 으로 사용, 절편 a(고정비, 음수) 버림 |
| ROE 54.9% 고정 | ROE = NPM×AT×FL 상수 → 매출과 무관, BV 가 커져도 ROE 불변 |
| BV 1.02조 → 1,018조 | BV 가 ROE×0.75 ≈ 41%/yr 복리, v5 선형 fade 는 줄어드는 스프레드에 폭증하는 BV 를 곱해 PV(RI) 155조 |

**v6 해결**
1. 클래스 1개 + `run_rim()` 1개. 패치 셀·클래스 셀 내부 테스트 제거 → 단일·엑셀·배치가 **같은 함수, 같은 MKT** 로 계산 (STEP 7 에서 단일 TP 와 배치 TP 일치 자동 검증).
2. Phase 1 = **매출 주도 clean surplus**: `NI = NPM(Sales)×Sales`, `RI = NI − Re×BV`, `BV += b×NI` → 자본이 쌓이면 ROE 가 자연히 하락.
3. NPM = 회귀 적합값 `b + 4a/S` (절편 포함) 을 **[min(TTM,중앙값)−10%p, max(TTM,중앙값)+5%p]** 밴드로 제한. 이연법인세 인식 분기는 `세전×(1−22%)` 로 정규화.
4. 매출예측 YoY 가드 [−50%, +60%] (ETS/SARIMA 폭주 방어), 실적과 겹친 예측 분기 제외.
5. Phase 2 = ROE → Re 선형 fade, **BV 성장 ≤ g(4%) < Re** → 항상 수렴. 마지막 해 RI=0 → TV=0.
6. 엑셀 `RI_Schedule` 는 수식으로 재계산, `Check` 셀(Excel TP − 모델 TP)=0. 같은 날 재실행 시 DB 는 해당 (ticker,date) 를 지우고 다시 씀.

**실행 순서** — STEP 1~4 는 1회, 이후 STEP 3 의 종목만 바꿔 STEP 5~6 반복

| STEP | 내용 |
|---|---|
| 1 | 라이브러리 임포트 · 경로 |
| 2 | 변수 선언 (모델 파라미터) |
| 3 | 종목 입력 (단일 / 배치) |
| 4 | 엔진: 4-1 DB·시장 / 4-2 RIM 엔진 / 4-3 출력 함수 |
| 5 | 단일 티커 평가 + 엑셀 저장 |
| 6 | 시각화 |
| 7 | 전체 종목 배치 → DB |
| 8 | 배치 결과 요약 · 시각화 |

## STEP 1 · 라이브러리 임포트

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 1 · 라이브러리 임포트 & 프로젝트 경로
# ═══════════════════════════════════════════════════════════════════
import sys, os, gc, time, warnings, traceback
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.font_manager
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display
from sqlalchemy import text
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# ── 프로젝트 루트 자동 감지 (데스크톱 / 노트북 겸용) ────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            break
    else:
        root = next((c for c in _CANDIDATE_ROOTS
                     if os.path.isdir(os.path.join(c, "DATA"))), None)
        if root is None:
            raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

PROJECT_ROOT = _setup_path()
print(f"[PATH] 프로젝트 루트 : {PROJECT_ROOT}")

# ── 프로젝트 공용 모듈 (수정하지 않음) ──────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    to_dg_ticker, get_pymysql_conn,
    load_korea_financials_wide, load_korea_marketcap_latest, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db,
)

# ── 한글 폰트 ───────────────────────────────────────────────────────
_installed = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _font in ("Malgun Gothic", "AppleGothic", "NanumGothic"):
    if _font in _installed:
        matplotlib.rcParams["font.family"] = _font
        break
matplotlib.rcParams["axes.unicode_minus"] = False


def log(tag, msg):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)


print("[OK] STEP 1 완료")

## STEP 2 · 변수 선언
평가 방법을 바꾸는 값은 이 셀에만 있다. 바꾼 뒤에는 STEP 4-2 부터 다시 실행.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 2 · 변수 선언 (모델 파라미터 — 평가 방법은 여기서만 바뀐다)
# ═══════════════════════════════════════════════════════════════════
MODEL_VERSION = "v6"

# ── DB 테이블 ───────────────────────────────────────────────────────
TABLE_FS        = "korea_fs_data_from_DG"
TABLE_FORECAST  = "korea_revenue_forecast_result"
TABLE_PRICE     = "KSE_Price"
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap"
TABLE_RESULT    = "korea_rim_valuation"
TABLE_QUALITY   = "korea_valuation_quality_log"

# ── 데이터 요건 ─────────────────────────────────────────────────────
FORECAST_HORIZON     = 8        # 매출 예측 분기 수 (= Phase 1 명시 예측 2년)
MIN_REVENUE_QUARTERS = 24       # 매출 실적 최소 분기 수 (universe 필터와 동일)
FC_MODEL_PRIORITY    = ("Ensemble", "SARIMA", "ETS", "Theta")

# ── 시장 / 자기자본비용 ─────────────────────────────────────────────
ERP_METHOD       = "damodaran_floor"   # 'damodaran_floor' | 'kospi_geo_10y'
DAMODARAN_ERP_KR = 0.07
GEO_FLOOR        = 0.07
RF_FALLBACK      = 0.035
BETA_YEARS       = 10
BETA_MIN_OBS     = 750
RD_DEFAULT       = 0.045                # EVA(WACC) 계산용 기본 타인자본비용

# ── 성장 가정 ───────────────────────────────────────────────────────
GDP_GROWTH         = 0.04    # 명목 장기성장률: plateau 매출 성장률 겸 Phase 2 BV 성장 상한
SALES_GROWTH_CAP   = 0.60    # 예측 연매출 YoY 상한 (ETS/SARIMA 폭주 방어, 1년차는 TTM 실적 대비)
SALES_GROWTH_FLOOR = -0.50   # 예측 연매출 YoY 하한

# ── 순이익률(NPM) 추정 ──────────────────────────────────────────────
NPM_LOOKBACK_Q        = 20     # NPM 추정에 쓰는 최근 분기 수
OLS_MIN_R2            = 0.30   # NI = a + b·Sales 회귀 채택 기준
OLS_MIN_SAMPLES       = 12
WINSORIZE_LIMITS      = (0.05, 0.95)
NPM_EXPANSION_MAX     = 0.05   # 예측 NPM 상한 = max(TTM, 중앙값) + 5%p
NPM_CONTRACTION_MAX   = 0.10   # 예측 NPM 하한 = min(TTM, 중앙값) - 10%p
NPM_ABS_CAP           = 0.60   # 예측 NPM 절대 상한
NORMALIZE_TAX_BENEFIT = True   # 세전이익>0 & 법인세<0(이연법인세 인식) 분기는 NI=세전×(1-법정세율)
STATUTORY_TAX         = 0.22

# ── 유보율 / 자본 ───────────────────────────────────────────────────
RETENTION_DEFAULT = 0.70
RETENTION_FLOOR   = -0.50
BV_RETENTION_CAP  = 0.75       # BV 누적에 쓰는 유보율 상한

# ── Moat 6-Tier (EVA spread OR ROE-Re spread → n_pos 미달 시 1단계 강등) ─
#   (label, eva_th, roe_re_th, n_pos_min, phase1_plateau_yr, phase2_fade_yr)
MOAT_TABLE = [
    ("Exceptional moat", 0.25, 1.50, 18, 5, 30),
    ("Wide moat",        0.15, 0.80, 16, 4, 28),
    ("Wide-Narrow moat", 0.10, 0.40, 13, 3, 25),
    ("Narrow moat",      0.06, 0.20, 10, 2, 20),
    ("Some moat",        0.03, 0.05,  7, 1, 15),
    ("No moat",          None, None,  0, 0, 10),
]
MOAT_UNKNOWN = ("Unknown (fallback)", 1, 8)    # (label, plateau_yr, fade_yr)

# ── 재무 항목 키 / 단위 ─────────────────────────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds", "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]
FS_UNIT_MULTIPLIER        = 1_000        # 천원 → 원
MARKETCAP_UNIT_MULTIPLIER = 1_000_000    # 백만원 → 원

# ── Sanity flag (값은 바꾸지 않고 경고만) ───────────────────────────
PB_FLAG     = 15.0     # 내재 P/B (IV/BV0) 경고 기준
UPSIDE_FLAG = 300.0    # Upside(%) 경고 기준

print(f"[OK] STEP 2 완료  model={MODEL_VERSION}  g={GDP_GROWTH:.1%}  "
      f"sales cap=+{SALES_GROWTH_CAP:.0%}  NPM cap=max(TTM,med)+{NPM_EXPANSION_MAX:.0%}p")

## STEP 3 · 종목 입력

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 3 · 종목 입력
# ═══════════════════════════════════════════════════════════════════
# ── 단일 종목 (STEP 5 · 6) ──────────────────────────────────────────
SINGLE_TICKER = "A326030"     # 평가할 종목 코드 (A + 6자리 또는 6자리)
EXPORT_EXCEL  = True          # True 면 STEP 5 에서 엑셀 저장
EXPORT_DIR    = Path(r"C:/reports") if os.name == "nt" else Path.home() / "reports"

# ── 배치 (STEP 7) ───────────────────────────────────────────────────
TICKER_START         = 0      # universe 슬라이스 시작
TICKER_END           = 9999   # universe 슬라이스 끝
RUN_TICKERS_OVERRIDE = []     # 비워두면 universe 사용, 채우면 이 종목만 실행 (예: ["A005930", "A000660"])
SKIP_DONE            = False  # True 면 체크포인트에 기록된 종목 건너뜀
SAVE_DB              = True   # 배치 결과 DB 저장 여부

CHECKPOINT_DIR = Path("_korea_rim_checkpoint")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DONE_PATH = CHECKPOINT_DIR / f"done_tickers_{MODEL_VERSION}.txt"
FAIL_PATH = CHECKPOINT_DIR / f"failed_tickers_{MODEL_VERSION}.txt"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[OK] STEP 3 완료  single={SINGLE_TICKER}  export={EXPORT_EXCEL} -> {EXPORT_DIR}")

## STEP 4 · 엔진 (1회 실행)
4-1 DB·시장 파라미터 → 4-2 RIM 엔진 → 4-3 출력 함수

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 4-1 · DB 연결 · 결과 테이블 · Universe · 시장 파라미터 (1회 실행)
# ═══════════════════════════════════════════════════════════════════
db_info = get_db_info()
engine  = get_engine(db_info)

CREATE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
  `id`              BIGINT      NOT NULL AUTO_INCREMENT,
  `date`            DATE        NOT NULL COMMENT '평가 실행일',
  `ticker`          VARCHAR(20) NOT NULL,
  `year_label`      VARCHAR(10)          COMMENT '예측 구간 종료 연도',
  `phase`           VARCHAR(10)          COMMENT 'ph1 / ph1e / ph2',
  `sales_forecast`  DOUBLE, `npm_forecast` DOUBLE, `asset_turnover` DOUBLE,
  `fin_leverage`    DOUBLE, `roe_forecast` DOUBLE, `re` DOUBLE, `ri_spread` DOUBLE,
  `bv_start`        DOUBLE, `ri` DOUBLE, `pv_ri` DOUBLE,
  `moat_label`      VARCHAR(30), `rho` DOUBLE, `n_phase2` INT, `g_terminal` DOUBLE,
  `pv_all_ri`       DOUBLE, `terminal_value` DOUBLE, `intrinsic_value` DOUBLE,
  `current_price`   DOUBLE, `upside_pct` DOUBLE, `target_price` DOUBLE,
  `beta_raw`        DOUBLE, `beta_blume` DOUBLE, `bv_source` VARCHAR(12),
  `revenue_quarters` INT, `forecast_model` VARCHAR(20), `forecast_date` DATE,
  `created_at`      DATETIME    DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`, `year_label`),
  INDEX idx_ticker (`ticker`), INDEX idx_date (`date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""
# v6 추가 컬럼 (비파괴 ALTER)
ALTER_SQLS = [
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `model_version` VARCHAR(10)",
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `period_label` VARCHAR(20)",
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `net_income_fc` DOUBLE",
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `implied_pb` DOUBLE",
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `npm_method` VARCHAR(12)",
    f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN IF NOT EXISTS `sanity_flag` VARCHAR(255)",
]
_conn = get_pymysql_conn(db_info)
try:
    with _conn.cursor() as cur:
        cur.execute(CREATE_SQL)
        for q in ALTER_SQLS:
            cur.execute(q)
    _conn.commit()
finally:
    _conn.close()
log("DB", f"결과 테이블 준비: {TABLE_RESULT}  host={db_info.get('host')}:{db_info.get('port')}")

# ── Universe ────────────────────────────────────────────────────────
universe_df = get_universe_with_min_history(
    db_info, min_quarters=MIN_REVENUE_QUARTERS, require_consecutive=False)
KOREA_TICKER_LIST = universe_df["ticker"].tolist()
log("DB", f"Universe (매출 >= {MIN_REVENUE_QUARTERS}Q): {len(KOREA_TICKER_LIST):,}개")

# ── 시장 파라미터: 단일/배치 평가가 반드시 같은 MKT 를 공유 ────────
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
KOSPI_PX = load_kospi_series(
    start_date=(datetime.today() - timedelta(days=365 * 12)).strftime("%Y-%m-%d"))
_mk = estimate_market_return(method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX, years=10,
                             damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
MKT = {"rf": float(RF), "e_rm": float(_mk["e_rm"]), "erp": float(_mk["e_rm"] - RF),
       "kospi": KOSPI_PX, "asof": datetime.now().strftime("%Y-%m-%d")}

print("=" * 60)
print(f"  Rf    = {MKT['rf']:7.3%}  ({RF_SOURCE})")
print(f"  ERP   = {MKT['erp']:7.3%}  ({_mk['note']})")
print(f"  E(Rm) = {MKT['e_rm']:7.3%}")
print(f"  KOSPI = {len(KOSPI_PX):,}거래일 ({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 4-2 · RIM 엔진 (클래스 1개 + 진입 함수 run_rim 1개, 패치 셀 없음)
# ═══════════════════════════════════════════════════════════════════
#  IV = BV0 + Σ PV(RI_t),   RI_t = NI_t − Re × BV_{t-1}      (TV = 0)
#
#  Phase 1 (명시 2년 + moat plateau)  : 매출 주도 clean surplus
#      Sales_t  ← 매출예측 (YoY 가드)      NI_t = NPM_t × Sales_t
#      BV_t     = BV_{t-1} + b × NI_t      (적자면 NI 전액 차감)
#  Phase 2 (moat 별 fade 기간 N)      : ROE → Re 선형 수렴
#      ROE_t    = Re + spread0 × (1 − t/N),  spread0 = NI_last×(1+g)/BV − Re
#      BV_t     = BV_{t-1} × (1 + min(g, ROE_t × b))   ← BV 성장 ≤ g < Re
#  마지막 해 spread = 0 → RI = 0 → 잔존가치 0 (CAP 종료 후 초과이익 소멸)
# ═══════════════════════════════════════════════════════════════════
_FC_RECENCY_CACHE: Dict[str, str] = {}


def _fc_recency_col(table_name: str) -> str:
    """updated_at 이 있으면 그것을, 없으면 created_at 을 예측 버전 기준으로 사용."""
    if table_name not in _FC_RECENCY_CACHE:
        sql = ("SELECT COLUMN_NAME FROM information_schema.COLUMNS "
               "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl")
        with engine.connect() as c:
            cols = {r[0].lower() for r in c.execute(
                text(sql), {"db": db_info["database"], "tbl": table_name}).fetchall()}
        _FC_RECENCY_CACHE[table_name] = "updated_at" if "updated_at" in cols else "created_at"
    return _FC_RECENCY_CACHE[table_name]


def load_revenue_forecast_safe(ticker: str, horizon: int = FORECAST_HORIZON):
    """최신 예측 버전 전체 분기 로드 + 3중 검증 (분기 수 / 연속성 / 상수 시계열).
    Returns (Series[천원], model, run_date)."""
    rc = _fc_recency_col(TABLE_FORECAST)
    for model in FC_MODEL_PRIORITY:
        p = {"t": ticker, "m": model, "ml": f"%\\_{model}"}
        with engine.connect() as c:
            row = c.execute(text(
                f"SELECT MAX(DATE({rc})) FROM {TABLE_FORECAST} "
                f"WHERE ticker=:t AND (indicator=:m OR indicator LIKE :ml)"), p).fetchone()
        if row is None or row[0] is None:
            continue
        run_date = str(row[0])
        df = pd.read_sql(text(
            f"SELECT date, value, {rc} AS run_ts FROM {TABLE_FORECAST} "
            f"WHERE ticker=:t AND (indicator=:m OR indicator LIKE :ml) AND DATE({rc})=:fd "
            f"ORDER BY date"), engine, params={**p, "fd": run_date})
        if df.empty:
            continue
        df["date"] = pd.to_datetime(df["date"])
        df["run_ts"] = pd.to_datetime(df["run_ts"])
        df = df.sort_values(["date", "run_ts"]).drop_duplicates("date", keep="last")
        fc = df.set_index("date")["value"].astype(float).sort_index()
        if len(fc) < horizon:
            raise ValueError(f"forecast {len(fc)}Q < {horizon}Q ({model} {run_date}) — 재예측 필요")
        fc = fc.iloc[:horizon]
        per = np.array([q.ordinal for q in fc.index.to_period("Q")])
        if (np.diff(per) != 1).any():
            raise ValueError(f"forecast 분기 불연속 ({model} {run_date}) — 재예측 필요")
        if fc.nunique() == 1:
            raise ValueError(f"forecast 상수 시계열 ({model} {run_date}) — 재예측 필요")
        return fc, model, run_date
    return pd.Series(dtype=float), "", None


class KoreaRIMModel:
    """한국 주식 RIM (매출 주도 clean-surplus + ROE 선형 fade). 계산은 run() 1회로 끝난다."""

    def __init__(self, ticker: str, mkt: Dict[str, Any], verbose: bool = False):
        self.ticker  = to_dg_ticker(ticker)
        self.mkt     = mkt
        self.verbose = verbose
        self.report  = DataQualityReport(ticker=self.ticker)
        self.flags: List[str] = []
        self.p: Dict[str, Any] = {}          # 추정된 입력값 전부 (진단/엑셀/DB 공용)
        self.fs: Optional[pd.DataFrame] = None
        self.sales_actual: Optional[pd.Series] = None
        self.sales_fc: Optional[pd.Series] = None
        self.schedule: Optional[pd.DataFrame] = None
        self.valuation: Optional[Dict[str, Any]] = None

    # ── 공통 유틸 ────────────────────────────────────────────────────
    def _log(self, msg):
        if self.verbose:
            log(self.ticker, msg)

    def _flag(self, field, note):
        self.flags.append(note)
        self.report.add(field, "warning", note=note)

    @staticmethod
    def _winsorize(s):
        s = s.dropna()
        if len(s) < 4:
            return s
        return s.clip(s.quantile(WINSORIZE_LIMITS[0]), s.quantile(WINSORIZE_LIMITS[1]))

    def _sum_cols(self, keys):
        out = pd.Series(0.0, index=self.fs.index)
        for k in keys:
            if k in self.fs.columns:
                out = out + self.fs[k].fillna(0)
        return out

    def _last(self, col):
        if col not in self.fs.columns:
            return np.nan
        s = self.fs[col].dropna()
        return float(s.iloc[-1]) if not s.empty else np.nan

    # ── 1. 데이터 ────────────────────────────────────────────────────
    def load_data(self):
        fs = load_korea_financials_wide(self.ticker, db_info, table_name=TABLE_FS,
                                        item_keys=None, fillna_zero=False)
        if fs is None or fs.empty:
            raise ValueError("FS 데이터 없음")
        fs = fs.sort_index()
        money = [c for c in fs.columns if c not in ("shares_treasury_adj", "shares_common")]
        fs[money] = fs[money] * FS_UNIT_MULTIPLIER

        miss = [k for k in ("revenue", "net_income", "total_equity")
                if k not in fs.columns or fs[k].dropna().empty]
        if miss:
            for k in miss:
                self.report.add(k, "missing", n_obs=0)
            raise ValueError(f"핵심 항목 결측: {miss}")

        # 이연법인세 인식 등 일회성 세금이익 정규화
        fs["ni_norm"] = fs["net_income"]
        n_adj = 0
        if NORMALIZE_TAX_BENEFIT and {"pretax_income", "tax_expense"} <= set(fs.columns):
            mask = (fs["pretax_income"] > 0) & (fs["tax_expense"] < 0) & fs["net_income"].notna()
            fs.loc[mask, "ni_norm"] = fs.loc[mask, "pretax_income"] * (1 - STATUTORY_TAX)
            n_adj = int(mask.sum())
            if n_adj:
                self.report.add("ni_norm", "ok", n_obs=n_adj,
                                note=f"세금이익 {n_adj}개 분기 → 세전×(1-{STATUTORY_TAX:.0%})")
        self.p["ni_norm_quarters"] = n_adj
        self.fs = fs

        actual = fs["revenue"].dropna()
        if len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual))
            raise ValueError(f"매출 실적 {len(actual)}Q < {MIN_REVENUE_QUARTERS}Q")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))
        self.sales_actual = actual

        fc, model, run_date = load_revenue_forecast_safe(self.ticker, FORECAST_HORIZON)
        if fc.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0)
            raise ValueError("매출 forecast 없음")
        fc = fc * FS_UNIT_MULTIPLIER
        last_q = actual.index.max().to_period("Q")
        overlap = int((fc.index.to_period("Q") <= last_q).sum())
        if overlap:
            fc = fc[fc.index.to_period("Q") > last_q]
            self._flag("revenue_forecast", f"forecast {overlap}Q 가 실적과 겹쳐 제외")
        if len(fc) < 4:
            raise ValueError("실적 이후 forecast < 4Q — 재예측 필요")
        self.report.add("revenue_forecast", "ok", n_obs=len(fc), note=f"model={model}")
        self.sales_fc = fc
        self.p.update(forecast_model=model, forecast_date=run_date,
                      revenue_quarters=len(actual), last_actual_q=str(last_q))
        self._log(f"Sales actual={len(actual)}Q (~{last_q})  forecast={len(fc)}Q ({model} {run_date})")

    # ── 2. 자기자본비용 / 세율 / 유보율 ──────────────────────────────
    def estimate_re(self):
        info = compute_beta_10y(self.ticker, db_info, kospi_series=self.mkt["kospi"],
                                years=BETA_YEARS, min_obs=BETA_MIN_OBS)
        beta_raw = info.get("beta_raw", np.nan)
        if beta_raw is None or np.isnan(beta_raw):
            beta_blume = 1.0
            self.report.add("beta", "fallback_median", n_obs=info.get("n_obs", 0),
                            value=1.0, note="베타 계산 실패 → 1.0")
        else:
            beta_blume = float(info["beta_blume"])
            self.report.add("beta", "ok", n_obs=info.get("n_obs", 0), value=beta_blume,
                            r2=info.get("r_squared"), note=f"β_raw={beta_raw:.3f}")
        re = self.mkt["rf"] + beta_blume * self.mkt["erp"]
        self.p.update(re=float(re), beta_raw=beta_raw, beta_blume=beta_blume,
                      beta_r2=info.get("r_squared", np.nan), beta_n=info.get("n_obs", 0))
        self._log(f"Re={re:.3%}  β_raw={beta_raw:.3f} β_blume={beta_blume:.3f}  "
                  f"Rf={self.mkt['rf']:.3%} ERP={self.mkt['erp']:.3%}")

    def estimate_tax_rate(self):
        fs = self.fs
        tax, note, status = STATUTORY_TAX, "법정세율 fallback", "fallback_zero"
        if {"pretax_income", "tax_expense"} <= set(fs.columns):
            df = fs[["pretax_income", "tax_expense"]].dropna()
            df = df[df["pretax_income"] > 0]
            if not df.empty:
                tax = float((df["tax_expense"] / df["pretax_income"]).clip(0, 0.40).median())
                note, status = f"median of {len(df)}Q", "ok"
        self.report.add("tax_rate", status, value=tax, note=note)
        self.p["tax_rate"] = tax

    def estimate_retention(self):
        fs = self.fs
        ni = fs["net_income"]
        ret, note, status = RETENTION_DEFAULT, "default", "fallback_zero"
        div = None
        if "dividends_paid" in fs.columns and (fs["dividends_paid"].abs() > 0).sum() >= 4:
            div, src = fs["dividends_paid"].abs(), "dividends_paid"
        elif "total_equity" in fs.columns:
            implied = (ni - fs["total_equity"].diff()).clip(lower=0)
            if (implied > 0).sum() >= 4:
                div, src = implied, "clean surplus"
        if div is not None and (ni > 0).sum() > 0:
            po = (div / ni.abs()).replace([np.inf, -np.inf], np.nan).iloc[-8:].dropna()
            if not po.empty:
                payout = float(po.clip(0, 3.0).median())
                ret = float(np.clip(1 - payout, RETENTION_FLOOR, 0.98))
                note, status = f"payout={payout:.1%} ({src})", "ok"
        self.report.add("retention", status, value=ret, note=note)
        b_bv = ret if ret < 0 else min(ret, BV_RETENTION_CAP)
        self.p.update(retention=ret, b_bv=b_bv, b_phase2=max(b_bv, 0.0))
        self._log(f"Retention={ret:.3f} → BV 유보율 b={b_bv:.3f}  ({note})")

    # ── 3. NPM 모형 ──────────────────────────────────────────────────
    def estimate_npm(self):
        df = pd.DataFrame({"rev": self.fs["revenue"], "ni": self.fs["ni_norm"]}).dropna()
        df = df[df["rev"] > 0].tail(NPM_LOOKBACK_Q)
        if len(df) < 4:
            m = dict(method="fallback", a=0.0, b=0.05, r2=np.nan, ttm=0.05, med=0.05,
                     anchor=0.05, lo=0.0, hi=0.10)
            self.report.add("npm", "fallback_zero", value=0.05, note="NPM 데이터 < 4Q")
        else:
            ttm = float(df["ni"].tail(4).sum() / df["rev"].tail(4).sum())
            med = float(self._winsorize(df["ni"] / df["rev"]).median())
            hi = min(NPM_ABS_CAP, max(ttm, med) + NPM_EXPANSION_MAX)
            lo = max(-0.50, min(ttm, med) - NPM_CONTRACTION_MAX)
            m = dict(method="blend", a=0.0, b=np.nan, r2=np.nan, ttm=ttm, med=med,
                     anchor=0.5 * ttm + 0.5 * med, lo=lo, hi=hi)
            if len(df) >= OLS_MIN_SAMPLES:
                slope, icpt, r, _, _ = stats.linregress(df["rev"], df["ni"])
                m["r2"] = float(r ** 2)
                if m["r2"] >= OLS_MIN_R2:
                    # NI_q = a + b·Sales_q  →  연간 NPM(S) = b + 4a/S  (영업레버리지 반영)
                    m.update(method="ols_fit", a=float(icpt), b=float(slope))
            self.report.add("npm", "ok" if m["method"] == "ols_fit" else "fallback_median",
                            n_obs=len(df), value=m["anchor"], r2=m["r2"],
                            note=f"{m['method']} TTM={ttm:.1%} med={med:.1%} band=[{lo:.1%},{hi:.1%}]")
        self.p["npm_model"] = m
        self._log(f"NPM {m['method']}: TTM={m['ttm']:.1%} med={m['med']:.1%} "
                  f"b={m['b']:.3f} a={m['a'] / 1e8:,.0f}억/Q R2={m['r2']:.2f}  "
                  f"band=[{m['lo']:.1%}, {m['hi']:.1%}]")

    def npm_at(self, annual_sales: float) -> Tuple[float, float]:
        """연매출 → (적용 NPM, 밴드 적용 전 NPM)."""
        m = self.p["npm_model"]
        raw = (m["b"] + 4 * m["a"] / annual_sales
               if m["method"] == "ols_fit" and annual_sales > 0 else m["anchor"])
        return float(np.clip(raw, m["lo"], m["hi"])), float(raw)

    # ── 4. EVA spread → Moat → 기간 ──────────────────────────────────
    def _wacc_simple(self):
        re, tax, fs = self.p["re"], self.p["tax_rate"], self.fs
        rd = RD_DEFAULT
        if "interest_expense" in fs.columns:
            td = self._sum_cols(DEBT_KEYS)
            td_avg = (td + td.shift(1)) / 2
            ie = fs["interest_expense"].abs()
            ok = (td_avg > 0) & ie.notna()
            if ok.sum() >= 2:
                rd = float((ie[ok] / td_avg[ok]).clip(0, 0.20).median())
        mc, _ = load_korea_marketcap_latest(self.ticker, db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            return re
        e = mc * MARKETCAP_UNIT_MULTIPLIER
        d = float(sum(self._last(k) for k in DEBT_KEYS if not np.isnan(self._last(k))))
        return float(np.clip(re * e / (e + d) + rd * (1 - tax) * d / (e + d), 0.04, 0.25))

    def classify_moat(self):
        fs, re = self.fs, self.p["re"]
        wacc = self._wacc_simple()
        eva_spread, roic, n_pos = np.nan, np.nan, 0
        if "operating_income" in fs.columns:
            nopat_ttm = (fs["operating_income"] * (1 - self.p["tax_rate"])).rolling(4).sum()
            ic = fs["total_equity"].fillna(0) + self._sum_cols(DEBT_KEYS) - self._sum_cols(CASH_KEYS)
            roic_s = (nopat_ttm / ic.where(ic > 0)).dropna()
            if not roic_s.empty:
                roic = float(roic_s.iloc[-1])
                eva_spread = roic - wacc
                n_pos = int(((roic_s - wacc).tail(20) > 0).sum())
        roe_re = np.nan
        df = self.fs[["ni_norm", "total_equity"]].dropna()
        df = df[df["total_equity"] > 0]
        if len(df) >= 4:
            roe_ttm = df["ni_norm"].tail(4).sum() / df["total_equity"].tail(4).mean()
            roe_re = float(roe_ttm - re)

        if np.isnan(eva_spread) and np.isnan(roe_re):
            label, n1, n2 = MOAT_UNKNOWN
        else:
            es = -np.inf if np.isnan(eva_spread) else eva_spread
            rs = -np.inf if np.isnan(roe_re) else roe_re
            idx = next(i for i, (_, et, rt, *_r) in enumerate(MOAT_TABLE)
                       if et is None or es > et or rs > rt)
            if n_pos < MOAT_TABLE[idx][3] and idx < len(MOAT_TABLE) - 1:
                idx += 1                                   # 지속성 부족 → 1단계 강등
            label, _, _, _, n1, n2 = MOAT_TABLE[idx]
        self.p.update(wacc=wacc, roic=roic, eva_spread=eva_spread, roe_re_spread=roe_re,
                      n_pos=n_pos, moat_label=label, n_plateau=n1, n_phase2=n2)
        self._log(f"EVA: ROIC={roic:.2%} WACC={wacc:.2%} spread={eva_spread:+.2%} "
                  f"n_pos={n_pos}/20 ROE-Re={roe_re:+.2%} → {label} (plateau {n1}yr, fade {n2}yr)")

    # ── 5. RI 스케줄 ─────────────────────────────────────────────────
    def build_schedule(self):
        re, b_bv, b2 = self.p["re"], self.p["b_bv"], self.p["b_phase2"]
        g = float(min(GDP_GROWTH, re - 0.01))
        self.p["g_term"] = g

        # BV0 (음수면 IC 로 자본비용 부과, 가치합산은 실제 BV 기준)
        bv0 = self._last("total_equity")
        if np.isnan(bv0) or bv0 <= 0:
            debt = float(sum(self._last(k) for k in DEBT_KEYS if not np.isnan(self._last(k))))
            ta = self._last("total_assets")
            basis = (0 if np.isnan(bv0) else bv0) + debt
            if basis <= 0:
                basis = ta * 0.40 if not np.isnan(ta) and ta > 0 else 1e11
            basis = max(basis, 1e10)
            bv_source = "IC_fallback"
            self._flag("bv0", f"BV0 {bv0 / 1e8:,.0f}억 ≤ 0 → IC {basis / 1e8:,.0f}억 기준 자본비용")
        else:
            basis, bv_source = bv0, "Equity"
        self.p.update(bv0=float(bv0), bv0_basis=float(basis), bv_source=bv_source)

        # 연간 매출 경로 (YoY 가드)
        fc = self.sales_fc
        prev = float(self.sales_actual.tail(4).sum())
        self.p["sales_ttm"] = prev
        years = []
        for k in range(len(fc) // 4):
            ch = fc.iloc[4 * k: 4 * k + 4]
            raw = float(ch.sum())
            s = raw
            if prev > 0:
                gr = raw / prev - 1
                grc = float(np.clip(gr, SALES_GROWTH_FLOOR, SALES_GROWTH_CAP))
                s = prev * (1 + grc)
                if abs(grc - gr) > 1e-9:
                    self._flag("sales_guard", f"Y{k + 1} 매출 YoY {gr:+.0%} → {grc:+.0%} 제한")
            years.append(dict(year=ch.index[-1].year, phase="ph1", sales_raw=raw, sales=s,
                              period=f"{ch.index[0].to_period('Q')}~{ch.index[-1].to_period('Q')}"))
            prev = s
        for e in range(1, self.p["n_plateau"] + 1):
            prev = prev * (1 + GDP_GROWTH)
            years.append(dict(year=years[-1]["year"] + 1, phase="ph1e", sales_raw=np.nan,
                              sales=prev, period="plateau"))

        rows, bv, npm_hold = [], basis, None
        for y in years:
            if y["phase"] == "ph1":
                npm, npm_raw = self.npm_at(y["sales"])
                npm_hold = npm
                if abs(npm - npm_raw) > 1e-9:
                    self._flag("npm_band", f"{y['period']} NPM {npm_raw:.1%} → {npm:.1%} (밴드)")
            else:
                npm, npm_raw = npm_hold, npm_hold
            ni = npm * y["sales"]
            ri = ni - re * bv
            rows.append(dict(y, npm=npm, npm_raw=npm_raw, net_income=ni,
                             roe=ni / bv if bv > 0 else np.nan, re=re,
                             ri_spread=(ni / bv - re) if bv > 0 else np.nan,
                             bv_start=bv, ri=ri))
            bv = bv + (b_bv * ni if ni > 0 else ni)

        # Phase 2 시작 ROE: 이익이 g 로 이어진다고 보고 새 BV 로 나눔 (이익 경로 연속)
        roe_last = rows[-1]["roe"]
        ni_last = rows[-1]["net_income"]
        spread0 = float(ni_last * (1 + g) / bv - re) if bv > 0 else 0.0
        n2 = self.p["n_phase2"]
        self.p["spread0"] = spread0
        for t in range(1, n2 + 1):
            spread = spread0 * (1 - t / n2) if bv > 0 else 0.0
            roe = re + spread
            rows.append(dict(year=rows[-1]["year"] + 1, phase="ph2", period=f"fade {t}/{n2}",
                             sales_raw=np.nan, sales=np.nan, npm=np.nan, npm_raw=np.nan,
                             net_income=roe * bv, roe=roe, re=re, ri_spread=spread,
                             bv_start=bv, ri=spread * bv))
            g_bv = min(g, roe * b2) if roe > 0 else roe
            bv = max(bv * (1 + g_bv), 0.0)

        sch = pd.DataFrame(rows)
        sch.insert(0, "t", np.arange(1, len(sch) + 1))
        sch["pv_ri"] = sch["ri"] / (1 + re) ** sch["t"]
        self.schedule = sch
        self._log(f"Schedule: Ph1={(sch.phase != 'ph2').sum()}yr Ph2={n2}yr  "
                  f"ROE_end Ph1={roe_last:.1%}  Ph2 ROE0={re + spread0:.1%} → Re  g={g:.1%}")

    # ── 6. 가치 ──────────────────────────────────────────────────────
    def value(self):
        p, sch = self.p, self.schedule
        pv_ri = float(sch["pv_ri"].sum())
        iv = p["bv0"] + pv_ri
        shares = np.nan
        for col in ("shares_treasury_adj", "shares_common"):
            if col in self.fs.columns:
                s = self.fs[col].dropna()
                s = s[s > 0]
                if not s.empty:
                    shares = float(s.iloc[-1])
                    break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0)
        if iv < 0:
            self._flag("sanity", f"IV {iv / 1e12:,.2f}조 < 0 → TP 0 처리")
        tp = max(iv, 0.0) / shares if shares and not np.isnan(shares) else np.nan
        cp = load_current_price(self.ticker, db_info, table_name=TABLE_PRICE)
        cp = float(cp) if cp is not None else np.nan
        if np.isnan(cp):
            self.report.add("current_price", "missing", n_obs=0)
        up = (tp / cp - 1) * 100 if (cp > 0 and not np.isnan(tp)) else np.nan
        pb = iv / p["bv0"] if p["bv0"] > 0 else np.nan
        if not np.isnan(pb) and pb > PB_FLAG:
            self._flag("sanity", f"내재 P/B {pb:.1f}x > {PB_FLAG:.0f}x")
        if not np.isnan(up) and up > UPSIDE_FLAG:
            self._flag("sanity", f"Upside {up:+.0f}% > {UPSIDE_FLAG:.0f}%")

        m = p["npm_model"]
        self.valuation = {
            "ticker": self.ticker, "model_version": MODEL_VERSION, "asof": self.mkt["asof"],
            "re": p["re"], "g_terminal": p["g_term"],
            "bv0": p["bv0"], "bv0_basis": p["bv0_basis"], "bv_source": p["bv_source"],
            "pv_ri": pv_ri, "pv_ri_ph1": float(sch.loc[sch.phase != "ph2", "pv_ri"].sum()),
            "pv_ri_ph2": float(sch.loc[sch.phase == "ph2", "pv_ri"].sum()),
            "terminal_value": 0.0, "pv_tv": 0.0, "tv_weight_pct": 0.0,
            "intrinsic_value": iv, "shares": shares, "target_price": tp,
            "current_price": cp, "upside_pct": up, "implied_pb": pb,
            "moat_label": p["moat_label"], "n_phase1": int((sch.phase != "ph2").sum()),
            "n_phase2": p["n_phase2"], "spread0": p["spread0"],
            "eva_spread": p["eva_spread"], "roic": p["roic"], "wacc": p["wacc"],
            "roe_re_spread": p["roe_re_spread"], "n_pos": p["n_pos"],
            "beta_raw": p["beta_raw"], "beta_blume": p["beta_blume"],
            "retention": p["retention"], "b_bv": p["b_bv"], "tax_rate": p["tax_rate"],
            "npm_method": m["method"], "npm_ttm": m["ttm"], "npm_med": m["med"],
            "revenue_quarters": p["revenue_quarters"], "forecast_model": p["forecast_model"],
            "forecast_date": p["forecast_date"],
            "sanity_flag": "; ".join(self.flags)[:255],
        }
        self._log(f"RIM  BV0={p['bv0'] / 1e12:.2f}조 PV(RI)={pv_ri / 1e12:.2f}조 "
                  f"IV={iv / 1e12:.2f}조 P/B={pb:.2f}x  TP={tp:,.0f}원 CP={cp:,.0f}원 "
                  f"Up={up:+.1f}%  [{p['moat_label']}]")

    # ── 7. 실행 (유일한 계산 경로) ──────────────────────────────────
    def run(self):
        self.load_data()
        self.estimate_re()
        self.estimate_tax_rate()
        self.estimate_retention()
        self.estimate_npm()
        self.classify_moat()
        self.build_schedule()
        self.value()
        return self

    # ── 8. DB 저장 (같은 ticker·date 는 지우고 다시 씀) ─────────────
    def save_to_db(self, run_date: Optional[str] = None) -> int:
        if self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation

        def n(x):
            try:
                f = float(x)
                return None if (np.isnan(f) or np.isinf(f)) else f
            except (TypeError, ValueError):
                return None

        common = {
            "date": run_date, "ticker": self.ticker, "moat_label": v["moat_label"],
            "rho": None, "n_phase2": int(v["n_phase2"]), "g_terminal": n(v["g_terminal"]),
            "pv_all_ri": n(v["pv_ri"]), "terminal_value": 0.0,
            "intrinsic_value": n(v["intrinsic_value"]), "current_price": n(v["current_price"]),
            "upside_pct": n(v["upside_pct"]), "target_price": n(v["target_price"]),
            "beta_raw": n(v["beta_raw"]), "beta_blume": n(v["beta_blume"]),
            "bv_source": v["bv_source"], "revenue_quarters": int(v["revenue_quarters"]),
            "forecast_model": v["forecast_model"], "forecast_date": v["forecast_date"],
            "model_version": MODEL_VERSION, "implied_pb": n(v["implied_pb"]),
            "npm_method": v["npm_method"], "sanity_flag": v["sanity_flag"] or None,
        }
        rows = []
        for r in self.schedule.itertuples():
            rows.append({**common,
                         "year_label": str(r.year), "phase": r.phase, "period_label": r.period,
                         "sales_forecast": n(r.sales), "npm_forecast": n(r.npm),
                         "asset_turnover": None, "fin_leverage": None,
                         "roe_forecast": n(r.roe), "re": n(r.re), "ri_spread": n(r.ri_spread),
                         "bv_start": n(r.bv_start), "ri": n(r.ri), "pv_ri": n(r.pv_ri),
                         "net_income_fc": n(r.net_income)})
        cols = list(rows[0].keys())
        sql = (f"INSERT INTO `{TABLE_RESULT}` ({', '.join(cols)}) VALUES "
               f"({', '.join('%(' + c + ')s' for c in cols)})")
        conn = get_pymysql_conn(db_info)
        try:
            with conn.cursor() as cur:
                cur.execute(f"DELETE FROM `{TABLE_RESULT}` WHERE ticker=%s AND date=%s",
                            (self.ticker, run_date))
                cur.executemany(sql, rows)
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()
        return len(rows)

    def summary_record(self) -> Dict[str, Any]:
        v = self.valuation
        keys = ["ticker", "target_price", "current_price", "upside_pct", "intrinsic_value",
                "bv0", "pv_ri", "implied_pb", "re", "moat_label", "n_phase1", "n_phase2",
                "spread0", "eva_spread", "npm_method", "bv_source", "sanity_flag"]
        return {k: v.get(k) for k in keys}


def run_rim(ticker: str, verbose: bool = False, save_db: bool = False,
            run_date: Optional[str] = None) -> KoreaRIMModel:
    """단일/배치 공용 진입점. 같은 MKT·같은 날짜면 TP 가 항상 동일하다."""
    m = KoreaRIMModel(ticker, MKT, verbose=verbose).run()
    if save_db:
        m.save_to_db(run_date)
    return m


print("[OK] STEP 4-2 완료  KoreaRIMModel / run_rim 정의")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 4-3 · 출력 함수 (진단 출력 / 시각화 / 엑셀) — 모델을 다시 돌리지 않는다
# ═══════════════════════════════════════════════════════════════════
def _jo(x):
    return "N/A" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x / 1e12:,.3f}조"


def print_diagnostics(m: KoreaRIMModel):
    v, p, sch = m.valuation, m.p, m.schedule
    npm = p["npm_model"]
    print("=" * 78)
    print(f"  {m.ticker}  Korea RIM {MODEL_VERSION}   (평가 기준일 {v['asof']})")
    print("=" * 78)
    print(f"[1] Re = Rf {m.mkt['rf']:.3%} + β_blume {v['beta_blume']:.3f} × ERP {m.mkt['erp']:.3%}"
          f" = {v['re']:.3%}   (β_raw={v['beta_raw']:.3f})")
    print(f"[2] NPM {npm['method']}: TTM={npm['ttm']:.1%}  median={npm['med']:.1%}  "
          f"band=[{npm['lo']:.1%}, {npm['hi']:.1%}]  (세금이익 정규화 {p['ni_norm_quarters']}Q)")
    print(f"    유보율 {v['retention']:.2f} → BV 유보율 b={v['b_bv']:.2f}   세율 {v['tax_rate']:.1%}")
    print(f"[3] Moat {v['moat_label']}: EVA spread {v['eva_spread']:+.1%} (ROIC {v['roic']:.1%} − "
          f"WACC {v['wacc']:.1%}), ROE−Re {v['roe_re_spread']:+.1%}, n_pos {v['n_pos']}/20")
    print(f"    Phase1 {v['n_phase1']}yr (명시 {int((sch.phase == 'ph1').sum())} + plateau "
          f"{int((sch.phase == 'ph1e').sum())})  /  Phase2 fade {v['n_phase2']}yr  "
          f"spread0={v['spread0']:+.1%} → 0,  BV 성장 ≤ g={v['g_terminal']:.1%}")
    print(f"[4] 매출 TTM 실적 {_jo(p['sales_ttm'])}  (마지막 실적 {p['last_actual_q']}, "
          f"예측 {v['forecast_model']} {v['forecast_date']})")

    view = sch.copy()
    for c in ("sales", "sales_raw", "net_income", "bv_start", "ri", "pv_ri"):
        view[c] = view[c] / 1e12
    view = view[["t", "year", "phase", "period", "sales_raw", "sales", "npm", "net_income",
                 "bv_start", "roe", "ri_spread", "ri", "pv_ri"]]
    ph2 = view[view.phase == "ph2"]
    show = pd.concat([view[view.phase != "ph2"], ph2.head(3), ph2.tail(2)]).drop_duplicates("t")
    print("\n[5] RI 스케줄 (금액 단위: 조원, Phase2 는 앞 3년·끝 2년만 표시)")
    fmt = {"sales_raw": "{:.3f}", "sales": "{:.3f}", "npm": "{:.1%}", "net_income": "{:.3f}",
           "bv_start": "{:.3f}", "roe": "{:.1%}", "ri_spread": "{:+.1%}", "ri": "{:+.3f}",
           "pv_ri": "{:+.3f}"}
    for c, f in fmt.items():
        show[c] = show[c].map(lambda x, f=f: "-" if pd.isna(x) else f.format(x))
    display(show.reset_index(drop=True))

    print(f"[6] 가치  BV0 {_jo(v['bv0'])} [{v['bv_source']}]  + PV(RI) Ph1 {_jo(v['pv_ri_ph1'])}"
          f" + Ph2 {_jo(v['pv_ri_ph2'])}  = IV {_jo(v['intrinsic_value'])}   "
          f"내재 P/B {v['implied_pb']:.2f}x")
    print(f"    TP = IV / {v['shares']:,.0f}주 = {v['target_price']:,.0f}원   "
          f"CP {v['current_price']:,.0f}원   Upside {v['upside_pct']:+.1f}%")
    if m.flags:
        print("[7] 경고")
        for f in m.flags:
            print(f"    - {f}")
    print("\n" + m.report.summary())


def plot_rim(m: KoreaRIMModel):
    v, sch, fs = m.valuation, m.schedule, m.fs
    re = v["re"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"{m.ticker}  Korea RIM {MODEL_VERSION}  [{v['moat_label']}]  "
                 f"TP {v['target_price']:,.0f}원 / CP {v['current_price']:,.0f}원 "
                 f"({v['upside_pct']:+.1f}%)", fontsize=13, fontweight="bold")

    # (0,0) ROE: 과거 TTM vs 예측 경로
    ax = axes[0, 0]
    h = fs[["ni_norm", "total_equity"]].dropna()
    h = h[h["total_equity"] > 0]
    roe_h = (h["ni_norm"].rolling(4).sum() / h["total_equity"].rolling(4).mean()).dropna().tail(20)
    x0 = sch["year"].iloc[0] - 1
    if not roe_h.empty:
        ax.plot(roe_h.index.year + (roe_h.index.month - 1) / 12, roe_h * 100,
                "o-", ms=3, color="#2980b9", label="과거 ROE (TTM, 정규화 NI)")
    for ph, sty, col, lab in (("ph1", "s-", "#27ae60", "Phase1 명시"),
                              ("ph1e", "s--", "#16a085", "Phase1 plateau"),
                              ("ph2", "^-", "#8e44ad", "Phase2 fade")):
        d = sch[sch.phase == ph]
        if not d.empty:
            ax.plot(d["year"], d["roe"] * 100, sty, ms=4, color=col, label=lab)
    ax.axhline(re * 100, color="#e74c3c", ls=":", label=f"Re {re:.1%}")
    ax.axvline(x0 + 0.5, color="gray", lw=0.8, ls="--")
    ax.set_title("ROE 경로 vs 자기자본비용")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # (0,1) RI 와 PV(RI)
    ax = axes[0, 1]
    colors = {"ph1": "#27ae60", "ph1e": "#16a085", "ph2": "#8e44ad"}
    ax.bar(sch["year"], sch["ri"] / 1e12, color=[colors[x] for x in sch.phase], alpha=0.75,
           label="RI")
    ax.plot(sch["year"], sch["pv_ri"] / 1e12, "k.-", lw=1, label="PV(RI)")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_title("잔여이익 (조원)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # (1,0) 가치 구성 vs 시가총액
    ax = axes[1, 0]
    parts = [("BV0", v["bv0"], "#3498db"), ("PV(RI) Ph1", v["pv_ri_ph1"], "#27ae60"),
             ("PV(RI) Ph2", v["pv_ri_ph2"], "#8e44ad")]
    bottom = 0.0
    for lab, val, col in parts:
        val_u = val / 1e12
        ax.bar(["Intrinsic"], [val_u], bottom=bottom, color=col, alpha=0.85,
               label=f"{lab} {val_u:,.2f}조")
        bottom += val_u
    if not np.isnan(v["current_price"]) and not np.isnan(v["shares"]):
        mc = v["current_price"] * v["shares"] / 1e12
        ax.bar(["Market Cap"], [mc], color="#e67e22", alpha=0.85, label=f"시가총액 {mc:,.2f}조")
    ax.set_title(f"가치 구성 (IV {v['intrinsic_value'] / 1e12:,.2f}조, P/B {v['implied_pb']:.2f}x)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # (1,1) 매출 · NPM (실적 + 예측)
    ax = axes[1, 1]
    sa = m.sales_actual.rolling(4).sum().dropna()
    ax.plot(sa.index.year + (sa.index.month - 1) / 12, sa / 1e12, color="#34495e",
            label="매출 TTM 실적")
    d1 = sch[sch.phase != "ph2"]
    ax.plot(d1["year"], d1["sales"] / 1e12, "s-", color="#27ae60", label="예측 연매출 (가드 후)")
    if d1["sales_raw"].notna().any():
        ax.plot(d1["year"], d1["sales_raw"] / 1e12, "x:", color="#c0392b", label="예측 원값")
    ax.set_title("매출 (조원) / NPM")
    ax.grid(alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(d1["year"], d1["npm"] * 100, "o--", color="#f39c12", label="적용 NPM")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.show()


# ── 엑셀: RI_Schedule 는 수식으로 재계산 → Excel TP 와 모델 TP 가 같은지 Check 셀로 검증 ──
_XF = "Arial"
_HDR = PatternFill("solid", start_color="003876")
_INP = PatternFill("solid", start_color="FFF9C4")
_THIN = Side(style="thin", color="D9D9D9")
_BOX = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
_F_WON, _F_PCT, _F_X = '#,##0;(#,##0);"-"', "0.00%", '0.00"x"'


def _xl_header(ws, row, labels):
    for j, lab in enumerate(labels, 1):
        c = ws.cell(row=row, column=j, value=lab)
        c.font = Font(name=_XF, bold=True, color="FFFFFF")
        c.fill = _HDR
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = _BOX


def _xl_put(ws, ref, value, fmt=None, inp=False, bold=False):
    c = ws[ref]
    c.value = value
    c.font = Font(name=_XF, bold=bold, color="0000FF" if inp else "000000")
    if inp:
        c.fill = _INP
    if fmt:
        c.number_format = fmt
    c.border = _BOX
    return c


def _xl_num(x):
    try:
        f = float(x)
        return None if (np.isnan(f) or np.isinf(f)) else f
    except (TypeError, ValueError):
        return x


def export_rim_excel(m: KoreaRIMModel, out_dir: Path) -> Path:
    v, p, sch = m.valuation, m.p, m.schedule
    wb = Workbook()

    # ── Sheet 1: Inputs (파란 글씨 = 입력, 바꾸면 RI_Schedule/TP 재계산) ──
    ws = wb.active
    ws.title = "Inputs"
    ws["A1"] = f"{m.ticker}  Korea RIM {MODEL_VERSION}  (평가일 {v['asof']}, 단위: 원)"
    ws["A1"].font = Font(name=_XF, size=14, bold=True, color="003876")
    items = [
        ("Re (자기자본비용)", v["re"], _F_PCT, True, "Rf + β_blume × ERP"),
        ("Rf", m.mkt["rf"], _F_PCT, False, "BOK 10Y"),
        ("ERP", m.mkt["erp"], _F_PCT, False, ERP_METHOD),
        ("β_blume", v["beta_blume"], "0.000", False, f"β_raw={v['beta_raw']:.3f}"),
        ("BV 유보율 b (Phase1)", p["b_bv"], _F_PCT, True, f"원 유보율 {p['retention']:.2f}, 상한 {BV_RETENTION_CAP}"),
        ("BV 유보율 b (Phase2)", p["b_phase2"], _F_PCT, True, "음수 유보율은 0 처리"),
        ("g (Phase2 BV 성장 상한)", p["g_term"], _F_PCT, True, "min(GDP, Re-1%)"),
        ("Phase2 fade 기간 N", p["n_phase2"], "0", True, v["moat_label"]),
        ("BV0 실제 자본총계", p["bv0"], _F_WON, True, v["bv_source"]),
        ("BV0 자본비용 기준", p["bv0_basis"], _F_WON, True, "음수 BV 면 IC"),
        ("주식수 (자사주 차감)", v["shares"], "#,##0", True, ""),
        ("현재 주가", v["current_price"], _F_WON, True, TABLE_PRICE),
        ("모델 적정주가 (Python)", v["target_price"], _F_WON, False, "검증용 고정값"),
    ]
    _xl_header(ws, 3, ["항목", "값", "비고"])
    for i, (lab, val, fmt, inp, note) in enumerate(items, start=4):
        _xl_put(ws, f"A{i}", lab)
        _xl_put(ws, f"B{i}", _xl_num(val), fmt, inp=inp)
        _xl_put(ws, f"C{i}", note)
    R = {k: f"Inputs!$B${i}" for i, k in enumerate(
        ["re", "rf", "erp", "beta", "b1", "b2", "g", "n2", "bv0", "bv0b", "sh", "cp", "tp_model"], start=4)}
    r0 = 4 + len(items) + 1
    _xl_header(ws, r0, ["추정 근거", "값", "비고"])
    npm = p["npm_model"]
    extra = [
        ("NPM 방식", npm["method"], None, f"R²={npm['r2']:.2f}"),
        ("NPM TTM / median", f"{npm['ttm']:.2%} / {npm['med']:.2%}", None, "세금이익 정규화 NI"),
        ("NPM band", f"[{npm['lo']:.2%}, {npm['hi']:.2%}]", None, "OLS 적합값을 이 범위로 제한"),
        ("Moat", v["moat_label"], None, f"EVA {v['eva_spread']:+.2%}, ROE-Re {v['roe_re_spread']:+.2%}, n_pos {v['n_pos']}"),
        ("매출예측", f"{v['forecast_model']} ({v['forecast_date']})", None, f"YoY 가드 [{SALES_GROWTH_FLOOR:+.0%}, {SALES_GROWTH_CAP:+.0%}]"),
        ("경고", v["sanity_flag"] or "-", None, ""),
    ]
    for i, (lab, val, fmt, note) in enumerate(extra, start=r0 + 1):
        _xl_put(ws, f"A{i}", lab)
        _xl_put(ws, f"B{i}", val, fmt)
        _xl_put(ws, f"C{i}", note)
    for col, w in zip("ABC", (28, 22, 60)):
        ws.column_dimensions[col].width = w

    # ── Sheet 2: RI_Schedule (수식) ─────────────────────────────────
    ws = wb.create_sheet("RI_Schedule")
    hdr = ["t", "연도", "Phase", "구간", "매출(가드후)", "NPM", "순이익", "BV 기초",
           "ROE", "Re", "Spread", "RI", "할인계수", "PV(RI)"]
    _xl_header(ws, 1, hdr)
    n = len(sch)
    last_ph1 = int((sch.phase != "ph2").sum()) + 1     # 마지막 Phase1 행 번호
    for i, r in enumerate(sch.itertuples(), start=2):
        _xl_put(ws, f"A{i}", int(r.t))
        _xl_put(ws, f"B{i}", str(r.year))
        _xl_put(ws, f"C{i}", r.phase)
        _xl_put(ws, f"D{i}", r.period)
        bv_prev = R["bv0b"] if i == 2 else None
        if r.phase != "ph2":
            _xl_put(ws, f"E{i}", _xl_num(r.sales), _F_WON, inp=True)
            _xl_put(ws, f"F{i}", _xl_num(r.npm), _F_PCT, inp=True)
            _xl_put(ws, f"G{i}", f"=E{i}*F{i}", _F_WON)
            _xl_put(ws, f"H{i}", f"={bv_prev}" if i == 2 else f"=H{i-1}+IF(G{i-1}>0,{R['b1']}*G{i-1},G{i-1})", _F_WON)
            _xl_put(ws, f"I{i}", f"=IF(H{i}>0,G{i}/H{i},0)", _F_PCT)
            _xl_put(ws, f"K{i}", f"=I{i}-J{i}", _F_PCT)
            _xl_put(ws, f"L{i}", f"=G{i}-J{i}*H{i}", _F_WON)
        else:
            k = i - last_ph1                      # Phase2 경과 연수
            if k == 1:
                bv_f = f"=H{i-1}+IF(G{i-1}>0,{R['b1']}*G{i-1},G{i-1})"
            else:
                bv_f = (f"=MAX(H{i-1}*(1+IF(I{i-1}>0,MIN({R['g']},I{i-1}*{R['b2']}),I{i-1})),0)")
            _xl_put(ws, f"H{i}", bv_f, _F_WON)
            f2 = last_ph1 + 1                     # Phase2 첫 행
            _xl_put(ws, f"K{i}", f"=IF(H{i}>0,($G${last_ph1}*(1+{R['g']})/$H${f2}-$J${f2})*(1-{k}/{R['n2']}),0)", _F_PCT)
            _xl_put(ws, f"I{i}", f"=J{i}+K{i}", _F_PCT)
            _xl_put(ws, f"G{i}", f"=I{i}*H{i}", _F_WON)
            _xl_put(ws, f"L{i}", f"=K{i}*H{i}", _F_WON)
        _xl_put(ws, f"J{i}", f"={R['re']}", _F_PCT)
        _xl_put(ws, f"M{i}", f"=1/(1+J{i})^A{i}", "0.0000")
        _xl_put(ws, f"N{i}", f"=L{i}*M{i}", _F_WON)
    end = n + 1
    s = end + 2
    summary = [
        ("BV0 (실제)", f"={R['bv0']}", _F_WON),
        ("Σ PV(RI)", f"=SUM(N2:N{end})", _F_WON),
        ("내재가치 IV", f"=N{s}+N{s+1}", _F_WON),
        ("적정주가 (Excel)", f"=IF({R['sh']}>0,MAX(N{s+2},0)/{R['sh']},0)", _F_WON),
        ("현재 주가", f"={R['cp']}", _F_WON),
        ("Upside", f"=IF(N{s+4}>0,N{s+3}/N{s+4}-1,0)", _F_PCT),
        ("내재 P/B", f"=IF(N{s}>0,N{s+2}/N{s},0)", _F_X),
        ("Check: Excel TP - 모델 TP (입력 미변경 시 0)", f"=N{s+3}-{R['tp_model']}", _F_WON),
    ]
    for j, (lab, f, fmt) in enumerate(summary):
        _xl_put(ws, f"L{s+j}", lab, bold=True)
        _xl_put(ws, f"N{s+j}", f, fmt, bold=(j == 3))
    for col, w in zip("ABCDEFGHIJKLMN", (5, 7, 7, 16, 18, 9, 18, 20, 9, 8, 9, 18, 9, 18)):
        ws.column_dimensions[col].width = w
    ws.column_dimensions["L"].width = 40
    ws.freeze_panes = "E2"

    # ── Sheet 3: FS_History (분기 실적, 값) ─────────────────────────
    ws = wb.create_sheet("FS_History")
    cols = [c for c in ("revenue", "net_income", "ni_norm", "pretax_income", "tax_expense",
                        "operating_income", "total_equity", "total_assets") if c in m.fs.columns]
    _xl_header(ws, 1, ["분기"] + cols + ["NPM(정규화)", "ROE TTM"])
    fh = m.fs[cols].copy()
    fh["npm"] = fh["ni_norm"] / fh["revenue"].where(fh["revenue"] > 0)
    eq = fh["total_equity"].where(fh["total_equity"] > 0)
    fh["roe_ttm"] = fh["ni_norm"].rolling(4).sum() / eq.rolling(4).mean()
    for i, (dt, r) in enumerate(fh.iterrows(), start=2):
        ws.cell(row=i, column=1, value=str(dt.to_period("Q"))).font = Font(name=_XF)
        for j, c in enumerate(cols + ["npm", "roe_ttm"], start=2):
            cell = ws.cell(row=i, column=j, value=_xl_num(r[c]))
            cell.font = Font(name=_XF)
            cell.number_format = _F_PCT if c in ("npm", "roe_ttm") else _F_WON
    for j in range(1, len(cols) + 4):
        ws.column_dimensions[get_column_letter(j)].width = 18
    ws.freeze_panes = "B2"

    # ── Sheet 4: Quality ───────────────────────────────────────────
    ws = wb.create_sheet("Quality")
    qdf = m.report.to_dataframe()
    _xl_header(ws, 1, list(qdf.columns))
    for i, row in enumerate(qdf.itertuples(index=False), start=2):
        for j, val in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=_xl_num(val)).font = Font(name=_XF)
    for j in range(1, len(qdf.columns) + 1):
        ws.column_dimensions[get_column_letter(j)].width = 20

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{m.ticker}_RIM_{datetime.now():%Y%m%d}_{MODEL_VERSION}.xlsx"
    wb.save(path)
    return path


print("[OK] STEP 4-3 완료  print_diagnostics / plot_rim / export_rim_excel 정의")

## STEP 5 · 단일 티커 평가 + 엑셀 저장

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 5 · 단일 티커 평가 + 엑셀 저장   (run_rim 1회 → 같은 객체로 진단·엑셀·시각화)
# ═══════════════════════════════════════════════════════════════════
model = run_rim(SINGLE_TICKER, verbose=True, save_db=False)
print_diagnostics(model)
display(model.report.to_dataframe())

if EXPORT_EXCEL:
    xlsx_path = export_rim_excel(model, EXPORT_DIR)
    print(f"\n[EXCEL] 저장: {xlsx_path}")

SINGLE_TP = model.valuation["target_price"]      # STEP 7 배치와 일치 여부 검증용

## STEP 6 · 시각화

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 6 · 시각화 (STEP 5 의 model 객체를 그대로 사용 — 재계산 없음)
# ═══════════════════════════════════════════════════════════════════
plot_rim(model)

## STEP 7 · 전체 종목 배치

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 7 · 전체 종목 배치 평가 → DB 저장   (단일 평가와 동일한 run_rim 사용)
# ═══════════════════════════════════════════════════════════════════
RUN_TICKERS = list(RUN_TICKERS_OVERRIDE) or KOREA_TICKER_LIST[TICKER_START:TICKER_END]
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and DONE_PATH.exists():
    done_set = {l.strip() for l in DONE_PATH.read_text(encoding="utf-8").splitlines() if l.strip()}

records, reports = [], []
n_ok = n_fail = n_skip = 0
t0 = time.time()
log("BATCH", f"RIM {MODEL_VERSION} 배치 시작: {len(RUN_TICKERS):,}개  run_date={run_date}")
print("=" * 90)

for i, tk in enumerate(RUN_TICKERS, 1):
    head = f"[{i:>5}/{len(RUN_TICKERS)}] {tk:<8}"
    if tk in done_set:
        n_skip += 1
        continue
    try:
        m = run_rim(tk, verbose=False, save_db=SAVE_DB, run_date=run_date)
        rec = {**m.summary_record(), "status": "ok", "msg": ""}
        reports.append(m.report)
        n_ok += 1
        with open(DONE_PATH, "a", encoding="utf-8") as f:
            f.write(tk + "\n")
        flag = "  !" if rec["sanity_flag"] else ""
        print(f"{head} OK   TP={rec['target_price']:>12,.0f}  Up={rec['upside_pct']:+8.1f}%  "
              f"P/B={rec['implied_pb']:5.2f}x  Re={rec['re']:.2%}  [{rec['moat_label']}]{flag}",
              flush=True)
    except Exception as e:
        rec = {"ticker": to_dg_ticker(tk), "status": "fail", "msg": str(e)[:150]}
        n_fail += 1
        with open(FAIL_PATH, "a", encoding="utf-8") as f:
            f.write(f"{tk}\t{rec['msg']}\n")
        print(f"{head} FAIL {rec['msg']}", flush=True)
    records.append(rec)
    if i % 10 == 0:
        gc.collect()

el = time.time() - t0
print("=" * 90)
log("BATCH", f"완료 OK={n_ok} FAIL={n_fail} SKIP={n_skip}  {el:.0f}s ({el / max(n_ok + n_fail, 1):.1f}s/종목)")

if reports and SAVE_DB:
    nq = save_quality_report_to_db(reports, db_info, table_name=TABLE_QUALITY,
                                   run_date=run_date, model_name="RIM")
    log("QUALITY", f"품질 로그 {nq:,}건 → {TABLE_QUALITY}")

batch_df = pd.DataFrame(records)

# 단일 평가(STEP 5)와 배치 결과 일치 검증
_tk = to_dg_ticker(SINGLE_TICKER)
if "SINGLE_TP" in globals() and _tk in set(batch_df.get("ticker", [])):
    _bt = batch_df.loc[batch_df["ticker"] == _tk, "target_price"].iloc[0]
    _same = np.isclose(_bt, SINGLE_TP, rtol=1e-9)
    log("CHECK", f"{_tk} 단일 TP={SINGLE_TP:,.0f}  배치 TP={_bt:,.0f}  -> {'일치' if _same else '불일치'}")

## STEP 8 · 배치 결과 요약 · 시각화

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  STEP 8 · 배치 결과 요약 · 시각화 (STEP 7 결과, 없으면 DB 최신 평가일)
# ═══════════════════════════════════════════════════════════════════
if "batch_df" in globals() and not batch_df.empty:
    res = batch_df[batch_df["status"] == "ok"].copy()
    src = f"이번 배치 ({run_date})"
else:
    _conn = get_pymysql_conn(db_info)
    try:
        with _conn.cursor() as cur:
            cur.execute(f"""
                SELECT ticker, MAX(target_price) AS target_price, MAX(current_price) AS current_price,
                       MAX(upside_pct) AS upside_pct, MAX(implied_pb) AS implied_pb, MAX(re) AS re,
                       MAX(moat_label) AS moat_label, MAX(sanity_flag) AS sanity_flag, MAX(date) AS d
                FROM {TABLE_RESULT}
                WHERE date = (SELECT MAX(date) FROM {TABLE_RESULT} WHERE model_version=%s)
                  AND model_version=%s
                GROUP BY ticker""", (MODEL_VERSION, MODEL_VERSION))
            res = pd.DataFrame(cur.fetchall())
    finally:
        _conn.close()
    src = f"DB {res['d'].max() if not res.empty else '-'}"

if res.empty:
    print("[INFO] 결과 없음 — STEP 7 실행 필요")
else:
    for c in ("target_price", "current_price", "upside_pct", "implied_pb", "re"):
        res[c] = pd.to_numeric(res[c], errors="coerce")
    res = res.sort_values("upside_pct", ascending=False)
    cols = ["ticker", "target_price", "current_price", "upside_pct", "implied_pb", "re",
            "moat_label", "sanity_flag"]
    print(f"[{src}] OK {len(res):,}개   Upside 중앙값 {res['upside_pct'].median():+.1f}%   "
          f"P/B 중앙값 {res['implied_pb'].median():.2f}x   경고 {res['sanity_flag'].astype(bool).sum()}개")
    print("\n업사이드 상위 20"); display(res[cols].head(20))
    print("\n다운사이드 상위 20"); display(res[cols].tail(20))
    flagged = res[res["sanity_flag"].astype(bool)]
    if not flagged.empty:
        print(f"\nSanity 경고 종목 {len(flagged)}개"); display(flagged[cols])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Korea RIM {MODEL_VERSION} 배치 요약 — {src}", fontsize=13)
    axes[0].hist(res["upside_pct"].dropna().clip(-100, 300), bins=50, color="#1565C0", alpha=0.85)
    axes[0].axvline(0, color="black", lw=1)
    axes[0].set_title("Upside 분포 (%, -100~300 클립)")
    axes[1].hist(res["implied_pb"].dropna().clip(0, 10), bins=50, color="#42A5F5", alpha=0.85)
    axes[1].set_title("내재 P/B 분포 (0~10x 클립)")
    mc = res["moat_label"].value_counts()
    axes[2].barh(mc.index[::-1], mc.values[::-1], color="#003876", alpha=0.85)
    axes[2].set_title("Moat 분포")
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()